# 12.4 concurrent.futures

**Prerequisites:** 12.2 Threading, 12.3 Multiprocessing  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- One API over threads **and** processes - swap them by changing a class name
- `submit()` and `Future` vs `map()`
- ✅ **Exceptions are re-raised by `result()`** - the fix for 12.2's silent failures
- `as_completed()` - handle results as they finish, not in order
- Timeouts, cancellation, and 🔴 what cancellation cannot do
- Choosing `max_workers`
- 🔴 The deadlock you get by submitting to a pool from inside that pool

---

## One interface, two engines

**12.2** managed `Thread` objects by hand. **12.3** managed `Process` objects by hand. Both involved the same chores: start workers, distribute work, collect results, notice failures, shut down cleanly.

`concurrent.futures` does all of that, and — the important part — presents threads and processes through the **identical interface**:

```
    with ThreadPoolExecutor(max_workers=4) as pool:     I/O-bound  (12.2)
    with ProcessPoolExecutor(max_workers=4) as pool:    CPU-bound  (12.3)
         ^^^^^^^^^^^^^^^^^^
         change this one word; everything below is unchanged
```

That makes the CPU/IO decision from **12.1** a one-line change, and it makes it easy to *measure* both rather than guess.

> **This should be your default.** Reach for raw `Thread` or `Process` only when you need something the executor does not offer — a long-lived worker with its own state, or a daemon that outlives the pool.

## `submit()` and the `Future`

```
    future = pool.submit(func, arg)     schedules it, returns IMMEDIATELY
    future.result()                     blocks until done, returns the value
                                        - or RE-RAISES the exception
```

A **`Future`** is a receipt: a handle on a result that does not exist yet.

| Method | Does |
|---|---|
| `result(timeout=None)` | wait and return the value, or raise |
| `exception(timeout=None)` | wait and return the exception, or `None` |
| `done()` | finished (including cancelled or failed)? |
| `cancel()` | try to cancel — see the limits below |
| `add_done_callback(fn)` | run `fn(future)` when it finishes |

In [ ]:
import random
import time
from concurrent.futures import (
    ALL_COMPLETED,
    FIRST_EXCEPTION,
    ProcessPoolExecutor,
    ThreadPoolExecutor,
    as_completed,
    wait,
)


def fetch(name, delay):
    """Stands in for an HTTP call (11.5) or a database query (10.3)."""
    time.sleep(delay)
    return f"{name} ({delay:.2f}s)"


with ThreadPoolExecutor(max_workers=3) as pool:
    future = pool.submit(fetch, "config", 0.2)
    print("submitted; done yet?", future.done())
    print("the Future itself :", future)
    print("result()          :", future.result())     # blocks here
    print("done now?         :", future.done())

## ✅ Exceptions are no longer silent

This is the single best reason to prefer executors.

In **12.2**, an exception inside a thread printed a traceback to stderr and was **invisible to the caller** — `join()` returned normally and the program carried on as though nothing had happened.

With a `Future`, the exception is **stored and re-raised** when you call `result()`. It cannot be missed, because you have to ask for the result to use it.

🔴 The one trap: if you never call `result()` or `exception()`, the failure is still silent. Fire-and-forget submissions swallow errors just as thoroughly as raw threads.

In [ ]:
def flaky(name):
    if name == "bad":
        raise ValueError(f"cannot process {name!r}")
    return f"processed {name}"


with ThreadPoolExecutor(max_workers=2) as pool:
    good_future = pool.submit(flaky, "good")
    bad_future = pool.submit(flaky, "bad")

    print("both submitted, no error raised yet")
    print("  good:", good_future.result())

    # The exception surfaces HERE, in the calling thread
    try:
        bad_future.result()
    except ValueError as exc:
        print("  bad :", type(exc).__name__, "-", exc)

    # ...or inspect it without raising
    print("  exception() gives:", repr(bad_future.exception()))

print()
print("🔴 But nothing forces you to ask. A submit() whose result is never")
print("   collected fails as silently as a bare Thread (12.2).")

with ThreadPoolExecutor(max_workers=1) as pool:
    pool.submit(flaky, "bad")            # result never requested
print("   ^ that submission raised ValueError. Nothing reported it.")

## `map()` vs `submit()`

| | `map(func, iterable)` | `submit(func, *args)` |
|---|---|---|
| Returns | an iterator of **results** | a `Future` |
| Order | **input order**, always | whatever you do with the futures |
| Arguments | one iterable per parameter | any signature |
| Exceptions | raised when you reach that item | when you call `result()` |
| Lazy? | results are yielded in order, so a slow first item blocks the rest | no |

🔴 **`map()` preserving input order is the gotcha.** If item 1 takes ten seconds and item 2 takes one, you still cannot see item 2's result until item 1 arrives. When you want results *as they finish*, use `as_completed()`.

In [ ]:
JOBS = [("alpha", 0.30), ("beta", 0.05), ("gamma", 0.20), ("delta", 0.10)]

print("-- map(): results in INPUT order --")
started = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    for result in pool.map(fetch, [j[0] for j in JOBS], [j[1] for j in JOBS]):
        print(f"   {time.perf_counter() - started:5.2f}s  {result}")

print("\n-- as_completed(): results as they FINISH --")
started = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(fetch, name, delay): name for name, delay in JOBS}
    for future in as_completed(futures):
        name = futures[future]                 # the dict maps back to the input
        print(f"   {time.perf_counter() - started:5.2f}s  {future.result()}")

print("\nSame work, same total time - but as_completed lets you act on the")
print("fast results immediately instead of waiting behind 'alpha'.")

### `wait()` - when you need the set, not each one

`wait()` blocks until a condition over a whole set of futures is met, and returns `(done, not_done)`.

| `return_when` | Stops when |
|---|---|
| `ALL_COMPLETED` (default) | everything has finished |
| `FIRST_COMPLETED` | any one finishes |
| `FIRST_EXCEPTION` | any one raises (or all finish) |

`FIRST_EXCEPTION` is the useful one for fail-fast batch work: stop waiting the moment anything has gone wrong.

In [ ]:
def sometimes_fails(n):
    time.sleep(0.1 * n)
    if n == 2:
        raise RuntimeError(f"task {n} failed")
    return n


with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(sometimes_fails, n) for n in range(5)]
    done, not_done = wait(futures, return_when=FIRST_EXCEPTION, timeout=10)

    print(f"stopped waiting with {len(done)} done, {len(not_done)} outstanding")
    for future in done:
        if future.exception() is not None:
            print("   failed :", future.exception())
        else:
            print("   ok     :", future.result())

print("\n🔴 Note: the outstanding tasks were NOT stopped. wait() stops")
print("   *waiting*; it does not stop the work - exactly like join(timeout=)")
print("   in 12.2. The `with` block then blocks until they all finish anyway.")

## 🔴 Cancellation, and its hard limit

```
    future.cancel()   ->  True   it had not started; it never will
                      ->  False  it is already running - and will finish
```

**A running task cannot be cancelled.** This is the same fact as **12.2**: Python has no way to interrupt a thread from outside, because doing so would leave shared state in an unknown condition.

So `cancel()` only ever removes *queued* work. To stop work that is already running, the task has to cooperate — check a flag, exactly as **11.2**'s servers did.

> **Version note.** `Executor.shutdown(cancel_futures=True)` (3.9+) cancels everything still queued in one call — much tidier than looping over futures.

In [ ]:
import threading

stop_flag = threading.Event()


def cooperative(n):
    """Checks the flag, so it CAN be stopped part-way."""
    for step in range(10):
        if stop_flag.is_set():
            return f"task {n} stopped at step {step}"
        time.sleep(0.05)
    return f"task {n} completed"


pool = ThreadPoolExecutor(max_workers=2)
futures = [pool.submit(cooperative, n) for n in range(6)]
time.sleep(0.12)                                  # let the first two start

cancelled = sum(1 for f in futures if f.cancel())
print(f"cancel() succeeded for {cancelled} of {len(futures)} futures")
print("  ^ only the ones still QUEUED. The running two ignored it.\n")

stop_flag.set()                                   # the cooperative part
pool.shutdown(wait=True, cancel_futures=True)

for i, future in enumerate(futures):
    if future.cancelled():
        print(f"   {i}: cancelled before starting")
    else:
        print(f"   {i}: {future.result()}")

## Swapping the engine

The payoff. Identical code, one word changed, run against both workload types from **12.1**.

🔴 `ProcessPoolExecutor` needs its worker at module level, so this cell runs the comparison as a guarded script — for the reasons established in **12.3**.

In [ ]:
import shutil
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

WORK = Path(tempfile.mkdtemp(prefix="py124_"))
script = WORK / "swap.py"
script.write_text(textwrap.dedent('''
    import time
    from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor


    def cpu_work(n):
        total = 0
        for i in range(12_000_000):
            total += i % 7
        return total


    def io_work(n):
        time.sleep(0.4)
        return n


    def run(executor_cls, func, tasks=4):
        started = time.perf_counter()
        with executor_cls(max_workers=tasks) as pool:
            list(pool.map(func, range(tasks)))
        return time.perf_counter() - started


    if __name__ == "__main__":
        print(f"  {'workload':<12}{'threads':>10}{'processes':>12}   winner")
        print("  " + "-" * 46)
        for label, func in (("CPU-bound", cpu_work), ("I/O-bound", io_work)):
            t = run(ThreadPoolExecutor, func)
            p = run(ProcessPoolExecutor, func)
            winner = "processes" if p < t else "threads"
            print(f"  {label:<12}{t:>9.2f}s{p:>11.2f}s   {winner}")
'''), encoding="utf-8")

finished = subprocess.run([sys.executable, str(script)],
                          capture_output=True, text=True, timeout=300)
print(finished.stdout.rstrip() or finished.stderr[:600])

print()
print("CPU-bound  -> processes win: each has its own GIL (12.1, 12.3)")
print("I/O-bound  -> threads win  : no startup cost, and waiting is free")
print("\nOne word changed between them. That is the whole point of this API.")

## Choosing `max_workers`

| Executor | Default | Sensible choice |
|---|---|---|
| `ThreadPoolExecutor` | `min(32, cpu_count + 4)` | as many as the *remote* side tolerates |
| `ProcessPoolExecutor` | `cpu_count()` | `cpu_count()`, rarely more |

For threads the limit is almost never your CPU — it is what you are calling. An API that rate-limits at 10 requests/second is not helped by 50 threads; you will just collect 40 `429`s (**11.5**).

For processes, more workers than cores means they compete for the same cores while each still costs a full interpreter.

> **Version note.** `ProcessPoolExecutor` gained `max_tasks_per_child` in 3.11 — recycle a worker after N tasks, which is a blunt but effective cure for a worker that leaks memory.

In [ ]:
import os

cpus = os.cpu_count()
print(f"cpu_count()                    : {cpus}")
print(f"ThreadPoolExecutor default     : {min(32, cpus + 4)}")
print(f"ProcessPoolExecutor default    : {cpus}")

with ThreadPoolExecutor() as pool:
    print(f"actual thread pool max_workers : {pool._max_workers}")

print()
print("-- more threads is not always faster --")
for workers in (1, 2, 4, 8):
    started = time.perf_counter()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        list(pool.map(fetch, [f"job{i}" for i in range(8)], [0.1] * 8))
    print(f"   {workers:>2} workers, 8 tasks x 0.1s: {time.perf_counter() - started:.2f}s")

print("\n   Doubling workers halves the time only while there is work queued.")
print("   Past 8 there is nothing left to overlap.")

## 🔴 Do not submit to a pool from inside that pool

A classic self-inflicted deadlock:

```
    def outer(n):
        return pool.submit(inner, n).result()    <- occupies a worker, then
                                                   WAITS for a free worker
```

With `max_workers=2`, two `outer` tasks occupy both workers and each waits for an `inner` that can never be scheduled. Nothing raises. Everything stops.

**Fixes:** use a second pool for the inner work, restructure so the work is flat, or call `inner` directly rather than submitting it.

The cell below deadlocks on purpose, with a timeout so it cannot hang the notebook.

In [ ]:
nested_pool = ThreadPoolExecutor(max_workers=2)


def inner(n):
    time.sleep(0.05)
    return n * 2


def outer(n):
    # 🔴 occupies a worker, then waits for one to become free
    return nested_pool.submit(inner, n).result()


futures = [nested_pool.submit(outer, n) for n in range(2)]

try:
    for future in futures:
        print("   result:", future.result(timeout=2))
except TimeoutError:
    print("   TimeoutError - deadlocked, exactly as predicted")
    print("   both workers are busy waiting for a worker")

nested_pool.shutdown(wait=False, cancel_futures=True)

print("\n-- the fix: a separate pool for the inner work --")
with ThreadPoolExecutor(max_workers=2) as outer_pool, \
     ThreadPoolExecutor(max_workers=2) as inner_pool:

    def outer_fixed(n):
        return inner_pool.submit(inner, n).result()

    print("   results:", list(outer_pool.map(outer_fixed, range(2))))

In [ ]:
# ---- tidy up ----
shutil.rmtree(WORK, ignore_errors=True)
print("scratch removed:", not WORK.exists())

leftover = [t.name for t in threading.enumerate() if t is not threading.main_thread()]
print("threads still alive:", leftover or "none")
if leftover:
    print("  ^ the deliberately deadlocked pool from the cell above; it was")
    print("    shut down with wait=False, so its workers are still unwinding.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Submitting and never collecting the result.** The exception is stored in the `Future` and, if you never ask, discarded as silently as a bare thread.
2. 🔴 **Submitting to a pool from inside that pool.** Workers wait for workers. Deadlock, with no exception.
3. 🔴 **Expecting `cancel()` to stop running work.** It only removes queued tasks. Running tasks must cooperate via a flag.
4. **Expecting `map()` to yield results as they finish.** It yields in *input* order; use `as_completed()` for completion order.
5. **Using `ProcessPoolExecutor` with a function defined in a notebook cell.** It cannot be pickled - see **12.3**.
6. **Setting `max_workers` far above what the remote service allows.** More threads just means more rate-limit errors.
7. **Forgetting that `wait()` and `shutdown(wait=False)` do not stop work.** Neither does exiting the `with` block early - it blocks until everything finishes.
8. **Assuming an executor is always worth it.** For a handful of quick tasks the pool costs more than it saves (**12.3**).

## Best Practices

- Default to `concurrent.futures` over raw `Thread`/`Process`.
- Always retrieve results - or at least call `future.exception()` - so failures are seen.
- Use `as_completed()` when you want to act on results as they arrive.
- Map futures back to their inputs with a `{future: input}` dict.
- Use the executor as a context manager so shutdown is automatic.
- Pass `timeout=` to `result()` and `wait()`; never block indefinitely.
- Size `max_workers` by the *constraint* - cores for processes, the remote service for threads.
- Write the work as a plain function first, and confirm it is correct serially before adding a pool.

## Practice Exercises

Try these before moving on.

1. Take the `as_completed` example and add a progress counter that prints `3/8 done` as each result arrives.
2. 🔴 Submit ten tasks where the third raises, using `map()`. At what point does the exception surface, and how many of the others ran?
3. Write `run_with_retries(func, attempts=3)` that resubmits a failed task to the same pool, with exponential backoff (**11.5**).
4. Compare `wait(..., FIRST_COMPLETED)` in a loop against `as_completed()` for the same work. When is each clearer?
5. Use `add_done_callback` to log every completion. Which thread does the callback run on - and why does that matter if it touches shared state (**12.2**)?
6. Reproduce the nested-pool deadlock with `max_workers=4` and eight outer tasks. Does it still deadlock? What does that tell you about how such bugs escape testing?
7. Take the `Queue` producer/consumer from **12.2** and rewrite it with a `ThreadPoolExecutor`. Which is shorter, and which gives better control?